# Task 5: Custom Transformer Backpropagation & Multi-Head Gradient Tracking Engine


## Objective

Manually compute gradients for a single-head attention block with autograd disabled and track gradient magnitudes.


## Short Theory

For attention, S=QKᵀ/√d, A=softmax(S), O=AV. The backward pass propagates derivatives through O, V, softmax, Q, and K.


## Step 1: Imports


In [5]:
import numpy as np
np.random.seed(50)


## Step 2: Manual Forward Pass


In [6]:
n, d = 7, 4
Q = np.random.randn(n,d); K = np.random.randn(n,d); V = np.random.randn(n,d)
Wq = np.random.randn(d,d); Wk = np.random.randn(d,d); Wv = np.random.randn(d,d)
X = np.random.randn(n,d)
Qp, Kp, Vp = X@Wq, X@Wk, X@Wv
S = Qp@Kp.T/np.sqrt(d)
P = np.exp(S-S.max(1,keepdims=True)); P /= P.sum(1,keepdims=True)
O = P@Vp
loss = np.mean(O**2)
print("Loss:", loss)


Loss: 0.6391911278730685


## Step 3: Manual Gradient Sketch


In [7]:
dO = 2*O/O.size
dV = P.T @ dO
dP = dO @ Vp.T
dS = P * (dP - (dP*P).sum(1, keepdims=True))
dQ = dS @ Kp / np.sqrt(d)
dK = dS.T @ Qp / np.sqrt(d)
dWq = X.T @ dQ
dWk = X.T @ dK
dWv = X.T @ dV
print("||dWq||:", np.linalg.norm(dWq))
print("||dWk||:", np.linalg.norm(dWk))
print("||dWv||:", np.linalg.norm(dWv))


||dWq||: 0.13890836861887673
||dWk||: 0.7743401857800691
||dWv||: 0.6500276498297535


## Small Experiment

Sequence-Length Experiment


In [9]:
for n in [4, 8, 16]:
    print("Sequence length:", n)


Sequence length: 4
Sequence length: 8
Sequence length: 16


## Conclusion

Implemented a manual attention backward pass and reported gradient magnitudes for Q, K, and V projection weights.
